In [1]:
import mlflow
from ultralytics import YOLO, settings

In [2]:
settings.update({"mlflow": False})

In [4]:
mlflow.set_tracking_uri("http://localhost:5001/")
mlflow.set_experiment("yolov26-garbage-detection")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1776817374680, experiment_id='1', last_update_time=1776817374680, lifecycle_stage='active', name='yolov26-garbage-detection', tags={}, trace_location=None, workspace='default'>

In [4]:
params = {
    "imgsz": 704,
    "batch": 8,
    "data": "data.yaml",
    "cls_pw": 0.5
}

with mlflow.start_run():
    mlflow.set_tag("model", "yolov26n")
    mlflow.log_params(params)

    model = YOLO("yolo26n.pt")
    results = model.train(
        data=params["data"],
        epochs=3,
        imgsz=params["imgsz"],
        batch=params["batch"],
        cls_pw=params["cls_pw"],
        device="mps",
        save=True
    )
    
    best_model_path = str(results.save_dir) + "/weights/best.pt"
    model = YOLO(best_model_path)
    metrics = model.val()
    mAP = metrics.box.map  # map50-95
    mAP50 = metrics.box.map50  # map50
    mAP75 = metrics.box.map75  # map75
    mAPs = metrics.box.maps
    
    mlflow.log_metric("mAP", mAP)
    mlflow.log_metric("mAP50", mAP50)
    mlflow.log_metric("mAP75", mAP75)
    for index, value in enumerate(mAPs):
        mlflow.log_metrics({f"mAP_class_{index}": value})
    mlflow.log_artifact(best_model_path)

New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.39 🚀 Python-3.12.2 torch-2.10.0 MPS (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=704, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-10, nbs=64, nms=False, opset=None, opt

Corrupt JPEG data: 7548 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 5.3s/it 5:415.7ss
                   all       1015       1209      0.747      0.726      0.793      0.618

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        2/3       3.3G     0.8311      3.862   0.009531         19        704: 10% ━─────────── 27/274 1.9it/s 12:17<2:11
🏃 View run victorious-dove-645 at: http://localhost:5001/#/experiments/1/runs/9e45e788aeef457a9b73def0ad5f76fb
🧪 View experiment at: http://localhost:5001/#/experiments/1


KeyboardInterrupt: 

In [2]:
model = YOLO("best.pt")
model.export(format="onnx")

Ultralytics 8.4.39 🚀 Python-3.12.2 torch-2.10.0 CPU (Apple M1)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.9 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.71'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command '"/Users/muhamadanangmahrub/Downloads/ML Project/monitoring sampah yogyakarta/.venv/bin/python3" -m pip install --no-cache-dir "onnxslim>=0.1.71" ' returned non-zero exit status 1.
WARNING ⚠️ Retry 2/2 failed: Command '"/Users/muhamadanangmahrub/Downloads/ML Project/monitoring sampah yogyakarta/.venv/bin/python3" -m pip install --no-cache-dir "onnxslim>=0.1.71" ' returned non-zero exit status 1.
WARNING ⚠️ requirements: ❌ Command '"/Users/muhamadanangmahrub/Downloads/ML Project/monitoring sampah yogyakarta/.venv/bin/python3" -m pip install --no-cache-dir "onnxslim>=0.1.71" ' returned non-zero exit statu

'best.onnx'

In [12]:
mlflow.end_run()

In [13]:
with mlflow.start_run(run_id='95b0dfe817ff4ffdb87af63d4b6345fa'):
    mlflow.log_artifact("best.onnx")

🏃 View run loud-robin-816 at: http://localhost:5001/#/experiments/1/runs/95b0dfe817ff4ffdb87af63d4b6345fa
🧪 View experiment at: http://localhost:5001/#/experiments/1


In [14]:
mlflow.register_model(
    model_uri="mlflow-artifacts:/1/95b0dfe817ff4ffdb87af63d4b6345fa/artifacts/best.onnx",
    name="garbage-detector"
)

Registered model 'garbage-detector' already exists. Creating a new version of this model...
2026/04/24 06:16:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: garbage-detector, version 2
Created version '2' of model 'garbage-detector'.


<ModelVersion: aliases=[], creation_timestamp=1776986198165, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1776986198165, metrics=None, model_id=None, name='garbage-detector', params=None, run_id='', run_link='', source='mlflow-artifacts:/1/95b0dfe817ff4ffdb87af63d4b6345fa/artifacts/best.onnx', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>